In [17]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from backend/.env
backend_dir = Path('../backend').resolve()
env_path = backend_dir / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded environment variables from {env_path}")
else:
    print(f"⚠ Warning: .env file not found at {env_path}")

# Add backend to path
sys.path.insert(0, str(backend_dir))

from tailor_tom.latex_compiler import compile_latex
from tailor_tom.layout_analyzer import extract_line_metrics, check_quality
import logging

# Set up logging to see debug messages
logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')
logger = logging.getLogger('tailor_tom.layout_analyzer')
logger.setLevel(logging.WARNING)  # Set to WARNING to see our debug logs


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Loaded environment variables from /Users/ramiri/dev/projects/TailorTom/backend/.env


In [18]:
# Load main.tex
with open('../main.tex', 'r', encoding='utf-8') as f:
    latex_content = f.read()

print(f"Loaded LaTeX file: {len(latex_content)} characters")
print(f"First 500 chars:\n{latex_content[:500]}")


Loaded LaTeX file: 15922 characters
First 500 chars:
\documentclass{article}
\usepackage[cm]{fullpage}
\usepackage{color}
\usepackage{hyperref}
\usepackage[margin=0.5in]{geometry}
\usepackage{fontawesome}

\hypersetup{breaklinks=false,%
colorlinks=true,%
linkcolor=MyDarkBlue,%
urlcolor=MyDarkBlue}

\definecolor{MyDarkBlue}{rgb}{0,0.0,0.45}

%%%%%%%%%%%%%%%%%%%%%%%%%%
% Formatting parameters  %
%%%%%%%%%%%%%%%%%%%%%%%%%%

\newlength{\tabin}
\setlength{\tabin}{1em}
\newlength{\secsep}
\setlength{\secsep}{0.1cm}

\setlength{\parindent}{0in}
\setlengt


In [19]:
# Compile to PDF
print("Compiling LaTeX to PDF...")
compile_result = compile_latex(latex_content)

if compile_result.success:
    print(f"✓ Compilation successful: {len(compile_result.pdf_bytes)} bytes")
    print(f"  Pages: {compile_result.page_count}")
else:
    print(f"✗ Compilation failed: {compile_result.error_message}")
    raise Exception("Compilation failed")


Compiling LaTeX to PDF...
✓ Compilation successful: 120256 bytes
  Pages: 1


In [20]:
# Extract line metrics with LaTeX source
print("Extracting line metrics...")
bullet_metrics = extract_line_metrics(compile_result.pdf_bytes, latex=latex_content)
bullets = bullet_metrics.get("bullets", [])

print(f"\nFound {len(bullets)} bullets total")
print(f"\nLine count distribution:")
line_counts = {}
for bullet in bullets:
    count = bullet.get("line_count", 0)
    line_counts[count] = line_counts.get(count, 0) + 1

for count in sorted(line_counts.keys()):
    print(f"  {count} lines: {line_counts[count]} bullets")


Extracting line metrics...

Found 22 bullets total

Line count distribution:
  1 lines: 8 bullets
  2 lines: 14 bullets


In [21]:
# Check quality with max_bullet_lines=2 (same as in the logs)
max_bullet_lines = 2
target_pages = 1

print(f"Checking quality with max_bullet_lines={max_bullet_lines}, target_pages={target_pages}...")
quality_result = check_quality(
    pdf_bytes=compile_result.pdf_bytes,
    target_pages=target_pages,
    max_bullet_lines=max_bullet_lines,
    latex=latex_content
)

print(f"\nQuality Check Results:")
print(f"  Passes: {quality_result.passes}")
print(f"  Page count: {quality_result.page_count} (target: {quality_result.page_target})")
print(f"  Long bullets: {len(quality_result.long_bullets)}")
print(f"  Issues: {quality_result.issues_summary}")


Checking quality with max_bullet_lines=2, target_pages=1...

Quality Check Results:
  Passes: True
  Page count: 1 (target: 1)
  Long bullets: 0
  Issues: All criteria pass


In [22]:
# Show detailed breakdown of bullets with >2 lines
print("=" * 80)
print("BULLETS WITH >2 LINES (Should be flagged as long)")
print("=" * 80)

long_bullets = [b for b in bullets if b.get("line_count", 0) > max_bullet_lines]

for i, bullet in enumerate(long_bullets, 1):
    text_preview = bullet.get("text_preview", "")[:100]
    line_count = bullet.get("line_count", 0)
    lines_text = bullet.get("lines_text", [])
    y_start = bullet.get("y_start", 0)
    y_end = bullet.get("y_end", 0)
    
    print(f"\n--- Bullet {i} ({line_count} lines) ---")
    print(f"Text preview: {text_preview}...")
    print(f"Y position: {y_start:.1f} to {y_end:.1f} (height: {y_end - y_start:.1f}pt)")
    
    if lines_text:
        print(f"Detected lines ({len(lines_text)}):")
        for j, line in enumerate(lines_text, 1):
            words = line.split()
            print(f"  Line {j} ({len(words)} words, {len(line)} chars): {line[:80]}{'...' if len(line) > 80 else ''}")
    else:
        print("  (No lines_text available)")
    
    print("-" * 80)


BULLETS WITH >2 LINES (Should be flagged as long)


In [23]:
# Show ALL bullets with their line counts for comparison
print("=" * 80)
print("ALL BULLETS (for comparison)")
print("=" * 80)

for i, bullet in enumerate(bullets, 1):
    text_preview = bullet.get("text_preview", "")[:80]
    line_count = bullet.get("line_count", 0)
    lines_text = bullet.get("lines_text", [])
    
    status = "⚠️ LONG" if line_count > max_bullet_lines else "✓ OK"
    print(f"\n{status} Bullet {i}: {line_count} line(s)")
    print(f"  Text: {text_preview}...")
    
    if lines_text and len(lines_text) > 1:
        print(f"  Lines:")
        for j, line in enumerate(lines_text, 1):
            print(f"    {j}. {line[:70]}{'...' if len(line) > 70 else ''}")
    elif lines_text:
        print(f"  Single line: {lines_text[0][:70]}{'...' if len(lines_text[0]) > 70 else ''}")


ALL BULLETS (for comparison)

✓ OK Bullet 1: 1 line(s)
  Text: 3.95/4.0 GPA | Candidate for Bachelor of Science in Computer Science...
  Single line: 3.95/4.0 GPA | Candidate for Bachelor of Science in Computer Science

✓ OK Bullet 2: 2 line(s)
  Text: Relevant Coursework: Machine Learning, Object-Oriented Design, Algorithms and Da...
  Lines:
    1. Relevant Coursework: Machine Learning, Object-Oriented Design, Algorit...
    2. Design, Calculus 3, Linear Algebra, Differential Equations, Discrete M...

✓ OK Bullet 3: 2 line(s)
  Text: Designed and led HubSpot’s first production LLM-as-judge evaluation pipeline for...
  Lines:
    1. Designed and led HubSpot’s first production LLM-as-judge evaluation pi...
    2. success from 40% to 90% for 2,391 + daily AI agent users

✓ OK Bullet 4: 2 line(s)
  Text: Deployed the grader as Kubernetes jobs in a governed MLOps workflow that gates e...
  Lines:
    1. Deployed the grader as Kubernetes jobs in a governed MLOps workflow th...
    2. tool 

In [24]:
# Save PDF for visual inspection
output_path = '../output/debug_main.pdf'
with open(output_path, 'wb') as f:
    f.write(compile_result.pdf_bytes)

print(f"PDF saved to: {output_path}")
print(f"\nSummary:")
print(f"  Total bullets: {len(bullets)}")
print(f"  Bullets with >{max_bullet_lines} lines: {len(long_bullets)}")
print(f"  Quality check passes: {quality_result.passes}")
print(f"\nNext steps:")
print(f"  1. Open {output_path} to visually inspect the bullets")
print(f"  2. Compare visual line count with detected line count")
print(f"  3. Check if continuation detection is including extra lines")


PDF saved to: ../output/debug_main.pdf

Summary:
  Total bullets: 22
  Bullets with >2 lines: 0
  Quality check passes: True

Next steps:
  1. Open ../output/debug_main.pdf to visually inspect the bullets
  2. Compare visual line count with detected line count
  3. Check if continuation detection is including extra lines
